In [27]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 50)

TODAY = pd.Timestamp('2026-05-31')

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

from openpyxl import load_workbook

def overlay_to_excel(df, filepath, sheet_name, start_row=1, start_col=1, header=True):
    """Write df values into an existing sheet without deleting other content/formulas."""
    wb = load_workbook(filepath)
    if sheet_name not in wb.sheetnames:
        wb.create_sheet(sheet_name)
    ws = wb[sheet_name]

    r = start_row
    if header:
        for c_idx, col_name in enumerate(df.columns, start=start_col):
            ws.cell(row=r, column=c_idx, value=col_name)
        r += 1

    for _, row_data in df.iterrows():
        for c_idx, val in enumerate(row_data, start=start_col):
            if isinstance(val, (pd.Timestamp, np.datetime64)):
                val = pd.Timestamp(val).to_pydatetime()
            elif isinstance(val, (np.integer,)):
                val = int(val)
            elif isinstance(val, (np.floating,)):
                val = float(val)
            ws.cell(row=r, column=c_idx, value=val)
        r += 1

    wb.save(filepath)
    wb.close()

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    conn.cursor().execute("SELECT 1").fetchone()
print("ODBC connection OK")

ODBC connection OK


In [28]:
LOBS = ['KMX', 'FLD', 'FRN', 'ENT', 'STG', 'AN']

with open('ragu_by_hurdle_pool.sql', 'r') as f:
    ragu_query_template = f.read()

ragu_df = run_sql(ragu_query_template.format(lob='KMX'))
ragu_df['app_month'] = pd.to_datetime(ragu_df['app_month'])
ragu_df

,app_month,hurdle,loan_count,total_amt_financed,wavg_gross_loss_ragu,wavg_recovery_multiplier,wavg_ltv,wavg_apr,recovery_impact,ltv_impact,apr_impact,ragu_score
0,2025-01-01,1) Lower,2266,5.573930e+07,140.262476,0.574333,1.665288,0.251927,-0.393538,-1.182415,-0.207509,138.479013
1,2025-01-01,2) Higher,1960,4.773132e+07,141.316084,0.573191,1.683728,0.254117,-0.475468,-1.455901,-0.443418,138.941297
2,2025-02-01,1) Lower,3399,8.007147e+07,139.430092,0.572624,1.628973,0.249325,-0.507664,-0.625734,0.072665,138.369358
3,2025-02-01,2) Higher,3042,7.018386e+07,140.538223,0.570342,1.618922,0.251109,-0.667361,-0.467244,-0.119381,139.284237
4,2025-03-01,1) Lower,4335,1.043987e+08,139.787268,0.576991,1.615649,0.248803,-0.209202,-0.415209,0.128932,139.291789
5,2025-03-01,2) Higher,3851,9.128695e+07,140.820611,0.573763,1.617032,0.251915,-0.434440,-0.437220,-0.206235,139.742716
6,2025-04-01,1) Lower,2888,7.313442e+07,140.688411,0.579100,1.584496,0.248219,-0.063187,0.090843,0.191788,140.907855
7,2025-04-01,2) Higher,2485,6.204728e+07,141.857733,0.577184,1.620025,0.251552,-0.198756,-0.484722,-0.167102,141.007154
8,2025-05-01,1) Lower,2835,7.123695e+07,141.159484,0.585617,1.621442,0.249485,0.400302,-0.507155,0.055436,141.108067
9,2025-05-01,2) Higher,2392,6.002778e+07,142.475384,0.586944,1.627971,0.251708,0.500617,-0.610022,-0.183990,142.181990


In [29]:
tzero_query_template = """
SELECT
    DATE_TRUNC('month', ldcf.application_received_dtm) AS app_month,

    CASE
        WHEN ldcf.dealer_pricing_hurdle = 'mROA-KMX'
            THEN CASE WHEN lrn.roa >= 500 THEN '1) Lower' ELSE '2) Higher' END
        WHEN ldcf.dealer_pricing_hurdle = 'mROA-FLD'
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) > 750 THEN '1) Lower' ELSE '2) Higher' END
        WHEN ldcf.data_source_id = 101
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) < 500 THEN '2) Higher' ELSE '1) Lower' END
        WHEN COALESCE(lrn.maxltv, 0) < 500
            OR (COALESCE(lrn.maxltv, 0) < 850
                AND ldcf.dealer_pricing_hurdle = 'mROA-MCY'
                AND ldcf.application_received_dtm >= '2024-06-19')
            THEN '2) Higher'
        ELSE '1) Lower'
    END AS hurdle,

    SUM(ldcf.con_risk_model_score * ldcf.con_amount_financed_back)
        / NULLIF(SUM(ldcf.con_amount_financed_back), 0)    AS weighted_model_score,

    AVG(ldcf.disb_acquisition_fee_amt)                      AS avg_discount_dollars,

    SUM(ldcf.disb_acquisition_fee_amt)
        / NULLIF(SUM(ldcf.con_amount_financed_back), 0)    AS weighted_discount_pct,

    AVG(ldcf.con_apr)                                       AS straight_avg_apr,

    SUM(ldcf.con_apr * ldcf.con_amount_financed_back)
        / NULLIF(SUM(ldcf.con_amount_financed_back), 0)    AS weighted_apr

FROM los_deal_current_fact ldcf

INNER JOIN edwnpi.dealer_rollup_scd_current dru
    ON dru.dealer_number = ldcf.dealer_number

LEFT JOIN sandbox.loan_random_numbers lrn
    ON lrn.loan_id = ldcf.loan_id

WHERE ldcf.application_received_dtm >= '2025-01-01'
  AND aspect = 'CONTRACT'
  AND dru.riskdealergroup = '{lob}'

GROUP BY 1, 2
ORDER BY 1, 2
"""

In [30]:
loan_count_query_template = """
SELECT
    ldcf.account_number,
    DATE_TRUNC('month', ldcf.application_received_dtm) AS app_month,
    ldcf.con_amount_financed_back,
    ldcf.con_pti_back,
    ldcf.pb_monthly_income_gross_total AS income_pb,
    COALESCE(ldcf.cb_monthly_income_gross_total, 0) AS income_cb,
    ldcf.blackbook_history_adj_wholesale_amt AS bb_value,
    CASE
        WHEN ldcf.dealer_pricing_hurdle = 'mROA-KMX'
            THEN CASE WHEN lrn.roa >= 500 THEN '1) Lower' ELSE '2) Higher' END
        WHEN ldcf.dealer_pricing_hurdle = 'mROA-FLD'
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) > 750 THEN '1) Lower' ELSE '2) Higher' END
        WHEN ldcf.data_source_id = 101
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) < 500 THEN '2) Higher' ELSE '1) Lower' END
        WHEN COALESCE(lrn.maxltv, 0) < 500
            OR (COALESCE(lrn.maxltv, 0) < 850
                AND ldcf.dealer_pricing_hurdle = 'mROA-MCY'
                AND ldcf.application_received_dtm >= '2024-06-19')
            THEN '2) Higher'
        ELSE '1) Lower'
    END AS hurdle
FROM edwnpi.los_deal_current_fact ldcf
LEFT JOIN edwnpi.dealer_rollup_scd_current dru
    ON dru.dealer_number = ldcf.dealer_number
LEFT JOIN sandbox.loan_random_numbers lrn
    ON lrn.loan_id = ldcf.loan_id
WHERE ldcf.application_received_dtm >= '2025-01-01'
  AND ldcf.book_date >= '2020-01-01'
  AND ldcf.data_source_name != 'SPARTAN'
  AND ldcf.account_number != 90124841967
  AND dru.riskdealergroup = '{lob}'
"""

results = {}

for lob in LOBS:
    print(f'Processing {lob}...')

    # RAGU scores by hurdle pool
    ragu_lob = run_sql(ragu_query_template.format(lob=lob))
    ragu_lob['app_month'] = pd.to_datetime(ragu_lob['app_month'])

    # Loan count via weekly-method: lean query + credit filters + nunique
    lc_df = run_sql(loan_count_query_template.format(lob=lob))
    lc_df['app_month'] = pd.to_datetime(lc_df['app_month'])
    lc_df['total_income'] = lc_df['income_pb'].fillna(0) + lc_df['income_cb'].fillna(0)
    lc_df['bbltv'] = lc_df['con_amount_financed_back'] / lc_df['bb_value']
    lc_df = lc_df[lc_df['con_amount_financed_back'] <= 75000]
    lc_df = lc_df[lc_df['con_pti_back'] <= 0.6]
    lc_df = lc_df[lc_df['total_income'] <= 200000]
    lc_df = lc_df[(lc_df['bbltv'] <= 10.0) | (lc_df['bb_value'].isna()) | (lc_df['bb_value'] == 0)]
    loan_counts = lc_df.groupby(['app_month', 'hurdle'])['account_number'].nunique().reset_index()
    loan_counts.columns = ['app_month', 'hurdle', 'loan_count']

    # T0 data (model score, discount, apr)
    tzero_lob = run_sql(tzero_query_template.format(lob=lob))
    tzero_lob['app_month'] = pd.to_datetime(tzero_lob['app_month'])

    # Merge RAGU + T0, then replace loan_count with weekly-method counts
    merged_lob = ragu_lob.merge(tzero_lob, on=['app_month', 'hurdle'], how='left')
    merged_lob = merged_lob.drop(columns=['loan_count'])
    merged_lob = merged_lob.merge(loan_counts, on=['app_month', 'hurdle'], how='left')

    # T1 data
    t1_lob = run_sql(f"""
    SELECT *
    FROM sandbox.t_1
    WHERE loss_meeting_flag = 1
      AND effective_date >= '2025-01-01'
      AND roa_dealer_group = '{lob}'
    """)

    # T1 clean sheet
    t1_clean = t1_lob[['roa_dealer_group', 'effective_date', 'model_score', 'discount', 'apr',
                        'ragu_score', 'cnl', 'wal', 'apr_realization_factor', 'fee_income',
                        'pmt_proc_income', 'addl_income', 'var_cost', 'cost_of_debt']].copy()
    t1_clean['orig_yield'] = (t1_clean['apr'] * t1_clean['apr_realization_factor']
                              + t1_clean['discount'] / t1_clean['wal']
                              + t1_clean['fee_income']
                              + t1_clean['pmt_proc_income']
                              + t1_clean['addl_income'])
    t1_clean = t1_clean.sort_values('effective_date').reset_index(drop=True)

    # T0 orig yield: use previous month's T-1 assumptions with actual T0 apr and discount
    t1_for_t0 = t1_lob[['effective_date', 'ragu_score', 'cnl', 'apr_realization_factor', 'wal',
                         'var_cost', 'cost_of_debt',
                         'fee_income', 'pmt_proc_income', 'addl_income']].copy()
    t1_for_t0 = t1_for_t0.rename(columns={'ragu_score': 't1_ragu_score', 'cnl': 't1_cnl'})
    t1_for_t0['effective_date'] = pd.to_datetime(t1_for_t0['effective_date'])
    t1_for_t0['app_month'] = t1_for_t0['effective_date'] + pd.offsets.MonthBegin(1)

    ragu_clean = merged_lob[['app_month', 'hurdle', 'loan_count', 'ragu_score',
                              'weighted_model_score', 'weighted_discount_pct', 'weighted_apr']].copy()

    ragu_clean = ragu_clean.merge(t1_for_t0[['app_month', 't1_ragu_score', 't1_cnl',
                                              'apr_realization_factor', 'wal',
                                              'var_cost', 'cost_of_debt',
                                              'fee_income', 'pmt_proc_income', 'addl_income']],
                                  on='app_month', how='left')

    ragu_clean['orig_yield'] = (ragu_clean['weighted_apr'] * ragu_clean['apr_realization_factor']
                                + ragu_clean['weighted_discount_pct'] / ragu_clean['wal']
                                + ragu_clean['fee_income']
                                + ragu_clean['pmt_proc_income']
                                + ragu_clean['addl_income'])

    ragu_clean['t0_cnl'] = (ragu_clean['t1_cnl']
                             - (ragu_clean['ragu_score'] - ragu_clean['t1_ragu_score']) * 0.0065)

    ragu_clean['cnl_effect'] = ragu_clean['t0_cnl'] / ragu_clean['wal']

    ragu_clean['mroa'] = (ragu_clean['orig_yield']
                          - ragu_clean['cnl_effect']
                          - ragu_clean['var_cost']
                          - ragu_clean['cost_of_debt'])

    ragu_clean = ragu_clean.drop(columns=['t1_ragu_score', 't1_cnl', 'apr_realization_factor',
                                           'fee_income', 'pmt_proc_income', 'addl_income'])

    ragu_clean = ragu_clean[['app_month', 'hurdle', 'loan_count', 'ragu_score',
                              'weighted_model_score', 'weighted_discount_pct', 'weighted_apr',
                              't0_cnl', 'wal',
                              'orig_yield', 'cnl_effect', 'var_cost', 'cost_of_debt', 'mroa']]

    lower = ragu_clean[ragu_clean['hurdle'] == '1) Lower'].set_index('app_month')
    higher = ragu_clean[ragu_clean['hurdle'] == '2) Higher'].set_index('app_month')

    mroa_gap = higher['mroa'].values - lower['mroa'].values
    mmroa = (higher['mroa'] * higher['loan_count'] - lower['mroa'] * lower['loan_count']) / (higher['loan_count'] - lower['loan_count'])
    vol_change = higher['loan_count'] / lower['loan_count'] - 1

    ragu_clean['mroa_gap'] = np.nan
    ragu_clean['mmroa'] = np.nan
    ragu_clean['vol_change'] = np.nan
    ragu_clean.loc[ragu_clean['hurdle'] == '2) Higher', 'mroa_gap'] = mroa_gap
    ragu_clean.loc[ragu_clean['hurdle'] == '2) Higher', 'mmroa'] = mmroa.values
    ragu_clean.loc[ragu_clean['hurdle'] == '2) Higher', 'vol_change'] = vol_change.values

    results[lob] = {'mmroa': ragu_clean, 't1': t1_clean}
    print(f'  {lob} done: {len(ragu_clean)} rows')

print('\nAll LOBs processed.')

Processing KMX...
  KMX done: 34 rows
Processing FLD...
  FLD done: 34 rows
Processing FRN...
  FRN done: 34 rows
Processing ENT...
  ENT done: 34 rows
Processing STG...
  STG done: 34 rows
Processing AN...
  AN done: 34 rows

All LOBs processed.


In [32]:
output_file = 'ragu_profit_analysis.xlsx'

for lob in LOBS:
    overlay_to_excel(results[lob]['mmroa'], output_file, f'{lob}-mmroa')
    overlay_to_excel(results[lob]['t1'], output_file, f'T1 {lob}')

print(f'Exported to {output_file}')
print(f'Sheets: {[f"{lob}-mmroa" for lob in LOBS] + [f"T1 {lob}" for lob in LOBS]}')

Exported to ragu_profit_analysis.xlsx
Sheets: ['KMX-mmroa', 'FLD-mmroa', 'FRN-mmroa', 'ENT-mmroa', 'STG-mmroa', 'AN-mmroa', 'T1 KMX', 'T1 FLD', 'T1 FRN', 'T1 ENT', 'T1 STG', 'T1 AN']


In [ ]:
appr_query = """
SELECT
    DATE_TRUNC('week', app_date) AS week_start,
    AVG(str_appr_app * 1.00) AS str_appr_rate
FROM sandbox.kmx_approvals
WHERE app_date > '2024-01-01'
GROUP BY 1
ORDER BY 1
"""

appr_df = run_sql(appr_query)
appr_df['week_start'] = pd.to_datetime(appr_df['week_start'])
appr_df['month'] = appr_df['week_start'].dt.month
appr_df['day'] = appr_df['week_start'].dt.day

overall_avg = appr_df['str_appr_rate'].mean()

dec_mask = appr_df['month'] == 12
feb_mar_mask = ((appr_df['month'] == 2) & (appr_df['day'] >= 15)) | ((appr_df['month'] == 3) & (appr_df['day'] <= 15))

dec_avg = appr_df.loc[dec_mask, 'str_appr_rate'].mean()
feb_mar_avg = appr_df.loc[feb_mar_mask, 'str_appr_rate'].mean()

print(f"Overall avg straight approval rate: {overall_avg:.4f}")
print(f"December avg: {dec_avg:.4f} ({dec_avg / overall_avg:.3f}x vs overall)")
print(f"Feb 15 - Mar 15 avg: {feb_mar_avg:.4f} ({feb_mar_avg / overall_avg:.3f}x vs overall)")
print(f"\nDecember multiplier: {dec_avg / overall_avg:.4f}")
print(f"Feb15-Mar15 multiplier: {feb_mar_avg / overall_avg:.4f}")

# Approval rates by recency window, anchored to TODAY
SEASONALITY_FACTOR = 1.15

def in_feb_mar_window(week_start):
    """Check if a week falls within Feb 15 - Mar 15 of its year."""
    m, d = week_start.month, week_start.day
    return (m == 2 and d >= 15) or (m == 3 and d <= 15)

def seasonally_adjusted_avg(df_slice):
    """Adjust only the weeks in Feb 15 - Mar 15 by dividing by seasonality, then average."""
    adjusted = df_slice['str_appr_rate'].copy()
    seasonal_mask = df_slice['week_start'].apply(in_feb_mar_window)
    adjusted.loc[seasonal_mask] = adjusted.loc[seasonal_mask] / SEASONALITY_FACTOR
    return adjusted.mean(), seasonal_mask.sum()

last_1m_start = TODAY - pd.DateOffset(months=1)
last_3m_start = TODAY - pd.DateOffset(months=3)
last_6m_start = TODAY - pd.DateOffset(months=6)

mask_1m = (appr_df['week_start'] >= last_1m_start) & (appr_df['week_start'] < TODAY)
mask_3m = (appr_df['week_start'] >= last_3m_start) & (appr_df['week_start'] < TODAY)
mask_3to6m = (appr_df['week_start'] >= last_6m_start) & (appr_df['week_start'] < last_3m_start)

rate_1m = appr_df.loc[mask_1m, 'str_appr_rate'].mean()
rate_3m = appr_df.loc[mask_3m, 'str_appr_rate'].mean()
rate_3to6m = appr_df.loc[mask_3to6m, 'str_appr_rate'].mean()

adj_1m, adj_wks_1m = seasonally_adjusted_avg(appr_df.loc[mask_1m])
adj_3m, adj_wks_3m = seasonally_adjusted_avg(appr_df.loc[mask_3m])
adj_3to6m, adj_wks_3to6m = seasonally_adjusted_avg(appr_df.loc[mask_3to6m])

print(f"\n--- Approval Rates Anchored to {TODAY.date()} ---")
print(f"Last 1 month ({last_1m_start.date()} to {TODAY.date()}):")
print(f"  Raw: {rate_1m:.4f}  | Adj: {adj_1m:.4f}  ({adj_wks_1m} weeks seasonally adjusted)")
print(f"Last 3 months ({last_3m_start.date()} to {TODAY.date()}):")
print(f"  Raw: {rate_3m:.4f}  | Adj: {adj_3m:.4f}  ({adj_wks_3m} weeks seasonally adjusted)")
print(f"3 to 6 months ago ({last_6m_start.date()} to {last_3m_start.date()}):")
print(f"  Raw: {rate_3to6m:.4f}  | Adj: {adj_3to6m:.4f}  ({adj_wks_3to6m} weeks seasonally adjusted)")

Overall avg straight approval rate: 0.2010
December avg: 0.2030 (1.010x vs overall)
Feb 15 - Mar 15 avg: 0.2283 (1.136x vs overall)

December multiplier: 1.0098
Feb15-Mar15 multiplier: 1.1359

--- Approval Rates Anchored to 2026-05-31 ---
Last 1 month (2026-04-30 to 2026-05-31):
  Raw: 0.2225  | Adj: 0.2225  (0 weeks seasonally adjusted)
Last 3 months (2026-02-28 to 2026-05-31):
  Raw: 0.2400  | Adj: 0.2347  (2 weeks seasonally adjusted)
3 to 6 months ago (2025-11-30 to 2026-02-28):
  Raw: 0.2323  | Adj: 0.2274  (2 weeks seasonally adjusted)


In [ ]:
conv_query = """
WITH approved AS (
    SELECT
        DATE_TRUNC('week', app_date) AS week_start,
        SUM(str_appr_app) AS approved_apps
    FROM sandbox.kmx_approvals
    WHERE app_date > '2024-01-01'
    GROUP BY 1
),
booked AS (
    SELECT
        DATE_TRUNC('week', ldcf.application_received_dtm) AS week_start,
        COUNT(*) AS booked_contracts
    FROM los_deal_current_fact ldcf
    INNER JOIN edwnpi.dealer_rollup_scd_current dru
        ON dru.dealer_number = ldcf.dealer_number
    WHERE ldcf.application_received_dtm > '2024-01-01'
      AND aspect = 'CONTRACT'
      AND dru.riskdealergroup = 'KMX'
    GROUP BY 1
)
SELECT
    a.week_start,
    a.approved_apps,
    b.booked_contracts,
    b.booked_contracts * 1.0 / NULLIF(a.approved_apps, 0) AS conversion_rate
FROM approved a
LEFT JOIN booked b ON a.week_start = b.week_start
ORDER BY 1
"""

conv_df = run_sql(conv_query)
conv_df['week_start'] = pd.to_datetime(conv_df['week_start'])
conv_df['month'] = conv_df['week_start'].dt.month
conv_df['day'] = conv_df['week_start'].dt.day

conv_overall_avg = conv_df['conversion_rate'].mean()

conv_dec_mask = conv_df['month'] == 12
conv_feb_mar_mask = ((conv_df['month'] == 2) & (conv_df['day'] >= 15)) | ((conv_df['month'] == 3) & (conv_df['day'] <= 15))

conv_dec_avg = conv_df.loc[conv_dec_mask, 'conversion_rate'].mean()
conv_feb_mar_avg = conv_df.loc[conv_feb_mar_mask, 'conversion_rate'].mean()

print(f"Overall avg conversion rate: {conv_overall_avg:.4f}")
print(f"December avg: {conv_dec_avg:.4f} ({conv_dec_avg / conv_overall_avg:.3f}x vs overall)")
print(f"Feb 15 - Mar 15 avg: {conv_feb_mar_avg:.4f} ({conv_feb_mar_avg / conv_overall_avg:.3f}x vs overall)")
print(f"\nDecember multiplier: {conv_dec_avg / conv_overall_avg:.4f}")
print(f"Feb15-Mar15 multiplier: {conv_feb_mar_avg / conv_overall_avg:.4f}")

# Conversion rates by recency window with Feb-Mar seasonality adjustment of 1.25
CONV_SEASONALITY_FACTOR = 1.25

def conv_seasonally_adjusted_avg(df_slice):
    adjusted = df_slice['conversion_rate'].copy()
    seasonal_mask = df_slice['week_start'].apply(in_feb_mar_window)
    adjusted.loc[seasonal_mask] = adjusted.loc[seasonal_mask] / CONV_SEASONALITY_FACTOR
    return adjusted.mean(), seasonal_mask.sum()

conv_mask_1m = (conv_df['week_start'] >= last_1m_start) & (conv_df['week_start'] < TODAY)
conv_mask_3m = (conv_df['week_start'] >= last_3m_start) & (conv_df['week_start'] < TODAY)
conv_mask_3to6m = (conv_df['week_start'] >= last_6m_start) & (conv_df['week_start'] < last_3m_start)

conv_rate_1m = conv_df.loc[conv_mask_1m, 'conversion_rate'].mean()
conv_rate_3m = conv_df.loc[conv_mask_3m, 'conversion_rate'].mean()
conv_rate_3to6m = conv_df.loc[conv_mask_3to6m, 'conversion_rate'].mean()

conv_adj_1m, conv_adj_wks_1m = conv_seasonally_adjusted_avg(conv_df.loc[conv_mask_1m])
conv_adj_3m, conv_adj_wks_3m = conv_seasonally_adjusted_avg(conv_df.loc[conv_mask_3m])
conv_adj_3to6m, conv_adj_wks_3to6m = conv_seasonally_adjusted_avg(conv_df.loc[conv_mask_3to6m])

print(f"\n--- Conversion Rates Anchored to {TODAY.date()} (Feb-Mar adj: 1.25) ---")
print(f"Last 1 month ({last_1m_start.date()} to {TODAY.date()}):")
print(f"  Raw: {conv_rate_1m:.4f}  | Adj: {conv_adj_1m:.4f}  ({conv_adj_wks_1m} weeks seasonally adjusted)")
print(f"Last 3 months ({last_3m_start.date()} to {TODAY.date()}):")
print(f"  Raw: {conv_rate_3m:.4f}  | Adj: {conv_adj_3m:.4f}  ({conv_adj_wks_3m} weeks seasonally adjusted)")
print(f"3 to 6 months ago ({last_6m_start.date()} to {last_3m_start.date()}):")
print(f"  Raw: {conv_rate_3to6m:.4f}  | Adj: {conv_adj_3to6m:.4f}  ({conv_adj_wks_3to6m} weeks seasonally adjusted)")

Overall avg conversion rate: 0.1201
December avg: 0.1221 (1.016x vs overall)
Feb 15 - Mar 15 avg: 0.1439 (1.197x vs overall)

December multiplier: 1.0163
Feb15-Mar15 multiplier: 1.1974

--- Conversion Rates Anchored to 2026-05-31 (Feb-Mar adj: 1.25) ---
Last 1 month (2026-04-30 to 2026-05-31):
  Raw: 0.1140  | Adj: 0.1140  (0 weeks seasonally adjusted)
Last 3 months (2026-02-28 to 2026-05-31):
  Raw: 0.0815  | Adj: 0.0789  (2 weeks seasonally adjusted)
3 to 6 months ago (2025-11-30 to 2026-02-28):
  Raw: 0.1145  | Adj: 0.1100  (2 weeks seasonally adjusted)


## Conversion Rate & Full Call Rates (NONKMX)

In [ ]:
# Method 2: conversion_by_lob (time_received-based, from pricing_service_log)
method2_temp = """
SELECT loan_id,
       MIN(CASE WHEN function_name = 'workflow_pre_bureau_calculate' THEN created_utc_dtm END) AS first_app_processing,
       MIN(CASE WHEN function_name = 'workflow_calculate_contract' THEN created_utc_dtm END) AS first_contract_processing
INTO #time_received
FROM odsnpi.pricing_service_log psl
WHERE function_name IN ('workflow_pre_bureau_calculate', 'workflow_calculate_contract')
GROUP BY 1
"""

method2_query = """
SELECT ldcf.dealer_pricing_hurdle,
       FLOOR(DATE_DIFF('day', first_app_processing, SYSDATE) / 7) AS w,
       MIN(first_app_processing::DATE) AS first_day,
       COUNT(DISTINCT tr.loan_id) AS apps,
       COUNT(CASE WHEN DATEDIFF('day', first_app_processing, first_contract_processing) <= 7 THEN tr.loan_id END) AS first_week_cons,
       COUNT(CASE WHEN first_contract_processing IS NOT NULL THEN tr.loan_id END) AS contracts,
       first_week_cons * 1.000 / apps AS first_week_conversion,
       contracts * 1.000 / apps AS conversion
FROM #time_received tr
LEFT JOIN edwnpi.los_deal_current_fact ldcf
    ON ldcf.loan_id = tr.loan_id AND ldcf.aspect = 'APPLICATION'
WHERE tr.first_app_processing >= '2025-06-01'
  AND dealer_pricing_hurdle IN ('mROA-AN', 'mROA-ENT', 'mROA-FLD', 'mROA-FRN', 'mROA-KMX', 'mROA-MCY', 'mROA-STG')
  AND DATEDIFF('day', first_app_processing, SYSDATE) >= 7
GROUP BY 1, 2
ORDER BY 1, 2
"""

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    cursor = conn.cursor()
    cursor.execute(method2_temp)
    conn.commit()
    warnings.filterwarnings("ignore", category=UserWarning)
    method2_df = pd.read_sql_query(sql=method2_query, con=conn)
    warnings.filterwarnings("default", category=UserWarning)

method2_df['first_day'] = pd.to_datetime(method2_df['first_day'])
print(f"Conversion data: {len(method2_df)} rows")
print(f"Hurdles: {sorted(method2_df['dealer_pricing_hurdle'].dropna().unique())}")
method2_df.head(10)

Conversion data: 364 rows
Hurdles: ['mROA-AN', 'mROA-ENT', 'mROA-FLD', 'mROA-FRN', 'mROA-KMX', 'mROA-MCY', 'mROA-STG']


,dealer_pricing_hurdle,w,first_day,apps,first_week_cons,contracts,first_week_conversion,conversion
0,mROA-AN,1.0,2026-05-22,3919,258,271,0.065833,0.069150
1,mROA-AN,2.0,2026-05-15,3751,243,256,0.064783,0.068248
2,mROA-AN,3.0,2026-05-08,3457,255,270,0.073763,0.078102
3,mROA-AN,4.0,2026-05-01,3158,224,242,0.070931,0.076631
4,mROA-AN,5.0,2026-04-24,3974,267,287,0.067187,0.072219
5,mROA-AN,6.0,2026-04-17,3741,269,282,0.071906,0.075381
6,mROA-AN,7.0,2026-04-10,3175,211,231,0.066457,0.072756
7,mROA-AN,8.0,2026-04-03,2839,185,200,0.065164,0.070447
8,mROA-AN,9.0,2026-03-27,3083,217,233,0.070386,0.075576
9,mROA-AN,10.0,2026-03-20,3298,221,235,0.067010,0.071255


In [ ]:
method2_df['year'] = method2_df['first_day'].dt.year
method2_df['month'] = method2_df['first_day'].dt.month
method2_df['day'] = method2_df['first_day'].dt.day

CONV_DEC_FACTOR = 0.85
CONV_FM_FACTOR = 1.13

def _in_feb_mar(row):
    return (row['month'] == 2 and row['day'] >= 15) or (row['month'] == 3 and row['day'] <= 15)

fm_mask_conv = method2_df.apply(_in_feb_mar, axis=1)
dec_mask_conv = method2_df['month'] == 12
is_nonkmx = method2_df['dealer_pricing_hurdle'] != 'mROA-KMX'

method2_df['adj_conversion'] = method2_df['conversion'].copy()
method2_df.loc[fm_mask_conv & is_nonkmx, 'adj_conversion'] = (
    method2_df.loc[fm_mask_conv & is_nonkmx, 'conversion'] / CONV_FM_FACTOR)
method2_df.loc[dec_mask_conv & is_nonkmx, 'adj_conversion'] = (
    method2_df.loc[dec_mask_conv & is_nonkmx, 'conversion'] / CONV_DEC_FACTOR)

conv_monthly = method2_df.groupby(['dealer_pricing_hurdle', 'year', 'month']).apply(
    lambda g: pd.Series({
        'apps': g['apps'].sum(),
        'contracts': g['contracts'].sum(),
        'first_week_cons': g['first_week_cons'].sum(),
        'conversion': (g['conversion'] * g['apps']).sum() / g['apps'].sum(),
        'adj_conversion': (g['adj_conversion'] * g['apps']).sum() / g['apps'].sum(),
        'first_week_conversion': (g['first_week_conversion'] * g['apps']).sum() / g['apps'].sum(),
    })
).reset_index()

def weighted_avg_conv(df_slice, col='conversion'):
    if df_slice['apps'].sum() == 0:
        return np.nan
    return (df_slice[col] * df_slice['apps']).sum() / df_slice['apps'].sum()

m2_mask_1m = (method2_df['first_day'] >= last_1m_start) & (method2_df['first_day'] < TODAY)
m2_mask_3m = (method2_df['first_day'] >= last_3m_start) & (method2_df['first_day'] < TODAY)
m2_mask_3to6m = (method2_df['first_day'] >= last_6m_start) & (method2_df['first_day'] < last_3m_start)

print(f"{'='*70}")
print(f"  CONVERSION RATE BY LOB - Anchored to {TODAY.date()}")
print(f"  Seasonality: Dec={CONV_DEC_FACTOR}, Feb15-Mar15={CONV_FM_FACTOR} (non-KMX)")
print(f"{'='*70}")

conv_1m = weighted_avg_conv(method2_df.loc[m2_mask_1m])
conv_3m = weighted_avg_conv(method2_df.loc[m2_mask_3m])
conv_3to6m = weighted_avg_conv(method2_df.loc[m2_mask_3to6m])
adj_1m = weighted_avg_conv(method2_df.loc[m2_mask_1m], 'adj_conversion')
adj_3m = weighted_avg_conv(method2_df.loc[m2_mask_3m], 'adj_conversion')
adj_3to6m = weighted_avg_conv(method2_df.loc[m2_mask_3to6m], 'adj_conversion')

print(f"\n  OVERALL (all hurdles combined)")
print(f"    {'Window':<45} {'Raw':>8} {'Adjusted':>10}")
print(f"    {'Last 1 month':<45} {conv_1m:>8.4f} {adj_1m:>10.4f}")
print(f"    {'Last 3 months':<45} {conv_3m:>8.4f} {adj_3m:>10.4f}")
print(f"    {'3-6 months ago':<45} {conv_3to6m:>8.4f} {adj_3to6m:>10.4f}")

print(f"\n  BY HURDLE")
print(f"  {'-'*66}")
for hurdle in sorted(method2_df['dealer_pricing_hurdle'].dropna().unique()):
    h_df = method2_df[method2_df['dealer_pricing_hurdle'] == hurdle]
    h_1m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_1m].index)])
    h_3m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_3m].index)])
    h_3to6m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_3to6m].index)])
    a_1m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_1m].index)], 'adj_conversion')
    a_3m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_3m].index)], 'adj_conversion')
    a_3to6m = weighted_avg_conv(h_df.loc[h_df.index.intersection(method2_df.loc[m2_mask_3to6m].index)], 'adj_conversion')
    print(f"  {hurdle}:")
    print(f"    1m:  Raw: {h_1m:.4f}  Adj: {a_1m:.4f}   3m:  Raw: {h_3m:.4f}  Adj: {a_3m:.4f}   3-6m:  Raw: {h_3to6m:.4f}  Adj: {a_3to6m:.4f}")

output_file = 'ragu_profit_analysis.xlsx'
export_df = conv_monthly[['dealer_pricing_hurdle', 'year', 'month', 'apps', 'contracts',
                           'first_week_cons', 'conversion', 'adj_conversion', 'first_week_conversion']].copy()
export_df = export_df.sort_values(['dealer_pricing_hurdle', 'year', 'month']).reset_index(drop=True)

overlay_to_excel(export_df, output_file, 'Conversion by LOB')

print(f"\n  Exported to '{output_file}' sheet 'Conversion by LOB'")
export_df.head(15)

  CONVERSION RATE BY LOB - Anchored to 2026-05-31
  Seasonality: Dec=0.85, Feb15-Mar15=1.13 (non-KMX)

  OVERALL (all hurdles combined)
    Window                                             Raw   Adjusted
    Last 1 month                                    0.0335     0.0335
    Last 3 months                                   0.0343     0.0339
    3-6 months ago                                  0.0319     0.0319

  BY HURDLE
  ------------------------------------------------------------------
  mROA-AN:
    1m:  Raw: 0.0727  Adj: 0.0727   3m:  Raw: 0.0714  Adj: 0.0702   3-6m:  Raw: 0.0640  Adj: 0.0642
  mROA-ENT:
    1m:  Raw: 0.0222  Adj: 0.0222   3m:  Raw: 0.0272  Adj: 0.0265   3-6m:  Raw: 0.0322  Adj: 0.0323
  mROA-FLD:
    1m:  Raw: 0.0566  Adj: 0.0566   3m:  Raw: 0.0557  Adj: 0.0544   3-6m:  Raw: 0.0440  Adj: 0.0438
  mROA-FRN:
    1m:  Raw: 0.0363  Adj: 0.0363   3m:  Raw: 0.0336  Adj: 0.0330   3-6m:  Raw: 0.0263  Adj: 0.0262
  mROA-KMX:
    1m:  Raw: 0.0256  Adj: 0.0256   3m:  Ra

,dealer_pricing_hurdle,year,month,apps,contracts,first_week_cons,conversion,adj_conversion,first_week_conversion
0,mROA-AN,2025,6,11884.0,901.0,787.0,0.075816,0.075816,0.066223
1,mROA-AN,2025,7,10900.0,722.0,633.0,0.066239,0.066239,0.058073
2,mROA-AN,2025,8,13438.0,981.0,870.0,0.073002,0.073002,0.064742
3,mROA-AN,2025,9,9970.0,594.0,526.0,0.059579,0.059579,0.052758
4,mROA-AN,2025,10,11760.0,667.0,589.0,0.056718,0.056718,0.050085
5,mROA-AN,2025,11,10096.0,582.0,516.0,0.057647,0.057647,0.051109
6,mROA-AN,2025,12,8781.0,507.0,456.0,0.057738,0.067927,0.051930
7,mROA-AN,2026,1,11110.0,650.0,593.0,0.058506,0.058506,0.053375
8,mROA-AN,2026,2,15818.0,1130.0,1041.0,0.071438,0.066099,0.065811
9,mROA-AN,2026,3,13355.0,915.0,847.0,0.068514,0.064663,0.063422


In [ ]:
fc_query = """
SELECT pricing_hurdle_name,
       DATE_PART('year', rh.reqappdate) AS year,
       DATE_PART('month', rh.reqappdate) AS month,
       MIN(rh.reqappdate) AS first_date,
       COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END) AS apps,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) AS cons,
       SUM(booked) AS booked_cons,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS full_call,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) * 1.0000
           / NULLIF(COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END), 0) AS b2l,
       AVG(rehashed * 1.0000) AS app_rehash_rate,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_frac,
       SUM(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_count,
       AVG(active_apr) AS avg_apr
FROM edwnpi.crm_dealer_dim cdd
LEFT JOIN edwnpi.los_deal_current_fact lcdf ON cdd.dealer_number = lcdf.dealer_number
LEFT JOIN sandbox.rehashes rh ON lcdf.loan_id = rh.appid
WHERE cdd.current_version_flag = 1
  AND lcdf.application_received_dtm >= '2024-01-01'
  AND acall_amtfin IS NOT NULL
  AND pricing_hurdle_name IS NOT NULL
  AND pricing_hurdle_name != 'mROA-KMX'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

fc_df = run_sql(fc_query)
fc_df['year'] = fc_df['year'].astype(int)
fc_df['month'] = fc_df['month'].astype(int)
fc_df['first_date'] = pd.to_datetime(fc_df['first_date'])
print(f"Full call data: {len(fc_df)} rows, hurdles: {sorted(fc_df['pricing_hurdle_name'].unique())}")
fc_df.head(5)

Full call data: 180 rows, hurdles: ['mROA-AN', 'mROA-ENT', 'mROA-FLD', 'mROA-FRN', 'mROA-MCY', 'mROA-STG']


,pricing_hurdle_name,year,month,first_date,apps,cons,booked_cons,full_call,b2l,app_rehash_rate,low_disc_frac,low_disc_count,avg_apr
0,mROA-AN,2024,1,2024-01-01,8441,703,1364,0.5463,0.083284,0.0845,0.3537,2986.0,0.257515
1,mROA-AN,2024,2,2024-02-01,11246,1048,2078,0.5316,0.093189,0.0849,0.3418,3844.0,0.257080
2,mROA-AN,2024,3,2024-03-01,11544,1136,2253,0.5660,0.098406,0.0868,0.3662,4228.0,0.255504
3,mROA-AN,2024,4,2024-04-01,9795,915,1808,0.6016,0.093415,0.0913,0.4019,3937.0,0.254392
4,mROA-AN,2024,5,2024-05-01,9556,761,1524,0.5795,0.079636,0.1170,0.3554,3397.0,0.253865


In [ ]:
FC_FM_FACTOR = 1.05

fc_df['adj_low_disc_frac'] = fc_df['low_disc_frac'].copy()
fm_fc_mask = fc_df['month'].isin([2, 3])
fc_df.loc[fm_fc_mask, 'adj_low_disc_frac'] = fc_df.loc[fm_fc_mask, 'low_disc_frac'] / FC_FM_FACTOR

fc_mask_1m = (fc_df['first_date'] >= last_1m_start) & (fc_df['first_date'] < TODAY)
fc_mask_3m = (fc_df['first_date'] >= last_3m_start) & (fc_df['first_date'] < TODAY)
fc_mask_3to6m = (fc_df['first_date'] >= last_6m_start) & (fc_df['first_date'] < last_3m_start)

def weighted_avg_fc(df_slice, col='low_disc_frac'):
    if df_slice['apps'].sum() == 0:
        return np.nan
    return (df_slice[col] * df_slice['apps']).sum() / df_slice['apps'].sum()

print(f"{'='*70}")
print(f"  FULL CALL RATE (low_disc_frac) by LOB - Anchored to {TODAY.date()}")
print(f"  Seasonality: Feb15-Mar15={FC_FM_FACTOR} (non-KMX, no Dec adj)")
print(f"{'='*70}")

fc_1m = weighted_avg_fc(fc_df.loc[fc_mask_1m])
fc_3m = weighted_avg_fc(fc_df.loc[fc_mask_3m])
fc_3to6m = weighted_avg_fc(fc_df.loc[fc_mask_3to6m])
afc_1m = weighted_avg_fc(fc_df.loc[fc_mask_1m], 'adj_low_disc_frac')
afc_3m = weighted_avg_fc(fc_df.loc[fc_mask_3m], 'adj_low_disc_frac')
afc_3to6m = weighted_avg_fc(fc_df.loc[fc_mask_3to6m], 'adj_low_disc_frac')

print(f"\n  OVERALL (all nonKMX hurdles combined)")
print(f"    {'Window':<45} {'Raw':>8} {'Adjusted':>10}")
print(f"    {'Last 1 month':<45} {fc_1m:>8.4f} {afc_1m:>10.4f}")
print(f"    {'Last 3 months':<45} {fc_3m:>8.4f} {afc_3m:>10.4f}")
print(f"    {'3-6 months ago':<45} {fc_3to6m:>8.4f} {afc_3to6m:>10.4f}")

print(f"\n  BY HURDLE")
print(f"  {'-'*66}")
for hurdle in sorted(fc_df['pricing_hurdle_name'].unique()):
    h_df = fc_df[fc_df['pricing_hurdle_name'] == hurdle]
    h_1m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_1m].index)])
    h_3m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_3m].index)])
    h_3to6m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_3to6m].index)])
    a_1m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_1m].index)], 'adj_low_disc_frac')
    a_3m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_3m].index)], 'adj_low_disc_frac')
    a_3to6m = weighted_avg_fc(h_df[h_df.index.isin(fc_df.loc[fc_mask_3to6m].index)], 'adj_low_disc_frac')
    print(f"  {hurdle}:")
    print(f"    1m:  Raw: {h_1m:.4f}  Adj: {a_1m:.4f}   3m:  Raw: {h_3m:.4f}  Adj: {a_3m:.4f}   3-6m:  Raw: {h_3to6m:.4f}  Adj: {a_3to6m:.4f}")

output_file = 'ragu_profit_analysis.xlsx'
fc_export = fc_df[['pricing_hurdle_name', 'year', 'month', 'first_date', 'apps', 'cons',
                   'booked_cons', 'full_call', 'b2l', 'app_rehash_rate', 'low_disc_frac',
                   'adj_low_disc_frac', 'low_disc_count', 'avg_apr']].copy()
fc_export = fc_export.sort_values(['pricing_hurdle_name', 'year', 'month']).reset_index(drop=True)

overlay_to_excel(fc_export, output_file, 'Full Call by LOB')

print(f"\n  Exported to '{output_file}' sheet 'Full Call by LOB'")

  FULL CALL RATE (low_disc_frac) by LOB - Anchored to 2026-05-31
  Seasonality: Feb15-Mar15=1.05 (non-KMX, no Dec adj)

  OVERALL (all nonKMX hurdles combined)
    Window                                             Raw   Adjusted
    Last 1 month                                    0.2532     0.2532
    Last 3 months                                   0.2646     0.2596
    3-6 months ago                                  0.2543     0.2492

  BY HURDLE
  ------------------------------------------------------------------
  mROA-AN:
    1m:  Raw: 0.4501  Adj: 0.4501   3m:  Raw: 0.4181  Adj: 0.4120   3-6m:  Raw: 0.3498  Adj: 0.3427
  mROA-ENT:
    1m:  Raw: 0.2653  Adj: 0.2653   3m:  Raw: 0.2937  Adj: 0.2875   3-6m:  Raw: 0.3382  Adj: 0.3320
  mROA-FLD:
    1m:  Raw: 0.3099  Adj: 0.3099   3m:  Raw: 0.3264  Adj: 0.3196   3-6m:  Raw: 0.3208  Adj: 0.3143
  mROA-FRN:
    1m:  Raw: 0.1899  Adj: 0.1899   3m:  Raw: 0.2017  Adj: 0.1980   3-6m:  Raw: 0.1766  Adj: 0.1728
  mROA-MCY:
    1m:  Raw: 0.194

In [ ]:
# Rehash Rate (app vs booked contract) by LOB - from rehash_rate_booked.sql
with open('rehash_rate_booked.sql', 'r') as f:
    rehash_query = f.read()

rehash_df = run_sql(rehash_query)
rehash_df['year'] = rehash_df['year'].astype(int)
rehash_df['month'] = rehash_df['month'].astype(int)
rehash_df['first_date'] = pd.to_datetime(rehash_df['first_date'])
rehash_df['period'] = rehash_df['year'].astype(str) + '-' + rehash_df['month'].astype(str).str.zfill(2)

print(f"Rehash data: {len(rehash_df)} rows")
print(f"\nApp Rehash Rate by LOB and Month (since 2025-05):")
pivot_app = rehash_df.pivot_table(index='pricing_hurdle_name', columns='period', values='app_rehash_rate')
print(pivot_app.round(4).to_string())

print(f"\nContract (Booked) Rehash Rate by LOB and Month (since 2025-05):")
pivot_con = rehash_df.pivot_table(index='pricing_hurdle_name', columns='period', values='con_rehash_rate')
print(pivot_con.round(4).to_string())

Rehash data: 84 rows

App Rehash Rate by LOB and Month (since 2025-05):
period               2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06
pricing_hurdle_name                                                                                                                              
mROA-AN               0.0394   0.0444   0.0356   0.0381   0.0355   0.0368   0.0327   0.0319   0.0355   0.0342   0.0334   0.0291   0.0277   0.0273
mROA-ENT              0.0051   0.0063   0.0054   0.0060   0.0062   0.0068   0.0065   0.0057   0.0061   0.0063   0.0054   0.0050   0.0049   0.0024
mROA-FLD              0.0118   0.0132   0.0141   0.0157   0.0146   0.0141   0.0123   0.0127   0.0142   0.0134   0.0139   0.0171   0.0169   0.0135
mROA-FRN              0.0272   0.0266   0.0237   0.0257   0.0215   0.0201   0.0218   0.0218   0.0211   0.0238   0.0241   0.0248   0.0226   0.0189
mROA-MCY              0.0710   0.0673   0.0643   0.0

In [ ]:
# Rehash Rate: 1m, 3m, 3-6m averages by LOB (weighted by relevant population)
rh_mask_1m = (rehash_df['first_date'] >= last_1m_start) & (rehash_df['first_date'] < TODAY)
rh_mask_3m = (rehash_df['first_date'] >= last_3m_start) & (rehash_df['first_date'] < TODAY)
rh_mask_3to6m = (rehash_df['first_date'] >= last_6m_start) & (rehash_df['first_date'] < last_3m_start)

def weighted_avg_rate(df_slice, rate_col, weight_col):
    if df_slice[weight_col].sum() == 0:
        return np.nan
    return (df_slice[rate_col] * df_slice[weight_col]).sum() / df_slice[weight_col].sum()

print(f"{'='*70}")
print(f"  REHASH RATES by LOB - Anchored to {TODAY.date()}")
print(f"{'='*70}")

# Overall
app_rh_1m = weighted_avg_rate(rehash_df.loc[rh_mask_1m], 'app_rehash_rate', 'apps')
app_rh_3m = weighted_avg_rate(rehash_df.loc[rh_mask_3m], 'app_rehash_rate', 'apps')
app_rh_3to6m = weighted_avg_rate(rehash_df.loc[rh_mask_3to6m], 'app_rehash_rate', 'apps')

con_rh_1m = weighted_avg_rate(rehash_df.loc[rh_mask_1m], 'con_rehash_rate', 'cons')
con_rh_3m = weighted_avg_rate(rehash_df.loc[rh_mask_3m], 'con_rehash_rate', 'cons')
con_rh_3to6m = weighted_avg_rate(rehash_df.loc[rh_mask_3to6m], 'con_rehash_rate', 'cons')

print(f"\n  OVERALL (all nonKMX)")
print(f"    App Rehash Rate:")
print(f"      1m: {app_rh_1m:.4f}   3m: {app_rh_3m:.4f}   3-6m: {app_rh_3to6m:.4f}")
print(f"    Contract (Booked) Rehash Rate:")
print(f"      1m: {con_rh_1m:.4f}   3m: {con_rh_3m:.4f}   3-6m: {con_rh_3to6m:.4f}")

print(f"\n  BY HURDLE")
print(f"  {'-'*66}")
for hurdle in sorted(rehash_df['pricing_hurdle_name'].unique()):
    h_df = rehash_df[rehash_df['pricing_hurdle_name'] == hurdle]
    h_idx_1m = h_df.index.intersection(rehash_df.loc[rh_mask_1m].index)
    h_idx_3m = h_df.index.intersection(rehash_df.loc[rh_mask_3m].index)
    h_idx_3to6m = h_df.index.intersection(rehash_df.loc[rh_mask_3to6m].index)

    a1 = weighted_avg_rate(h_df.loc[h_idx_1m], 'app_rehash_rate', 'apps')
    a3 = weighted_avg_rate(h_df.loc[h_idx_3m], 'app_rehash_rate', 'apps')
    a36 = weighted_avg_rate(h_df.loc[h_idx_3to6m], 'app_rehash_rate', 'apps')
    c1 = weighted_avg_rate(h_df.loc[h_idx_1m], 'con_rehash_rate', 'cons')
    c3 = weighted_avg_rate(h_df.loc[h_idx_3m], 'con_rehash_rate', 'cons')
    c36 = weighted_avg_rate(h_df.loc[h_idx_3to6m], 'con_rehash_rate', 'cons')
    print(f"  {hurdle}:")
    print(f"    App:      1m: {a1:.4f}   3m: {a3:.4f}   3-6m: {a36:.4f}")
    print(f"    Contract: 1m: {c1:.4f}   3m: {c3:.4f}   3-6m: {c36:.4f}")

  REHASH RATES by LOB - Anchored to 2026-05-31

  OVERALL (all nonKMX)
    App Rehash Rate:
      1m: 0.0214   3m: 0.0217   3-6m: 0.0199
    Contract (Booked) Rehash Rate:
      1m: 0.2322   3m: 0.2271   3-6m: 0.2370

  BY HURDLE
  ------------------------------------------------------------------
  mROA-AN:
    App:      1m: 0.0277   3m: 0.0300   3-6m: 0.0339
    Contract: 1m: 0.1617   3m: 0.1638   3-6m: 0.2190
  mROA-ENT:
    App:      1m: 0.0049   3m: 0.0051   3-6m: 0.0061
    Contract: 1m: 0.0932   3m: 0.0805   3-6m: 0.0820
  mROA-FLD:
    App:      1m: 0.0169   3m: 0.0158   3-6m: 0.0134
    Contract: 1m: 0.1186   3m: 0.1216   3-6m: 0.1287
  mROA-FRN:
    App:      1m: 0.0226   3m: 0.0239   3-6m: 0.0224
    Contract: 1m: 0.3170   3m: 0.3259   3-6m: 0.3634
  mROA-MCY:
    App:      1m: 0.0888   3m: 0.0749   3-6m: 0.0648
    Contract: 1m: 0.3940   3m: 0.3154   3-6m: 0.3567
  mROA-STG:
    App:      1m: 0.0270   3m: 0.0277   3-6m: 0.0268
    Contract: 1m: 0.2405   3m: 0.2338   3-6m: 0